In [ ]:
import pandas as pd
import numpy as np
import seaborn as sb

In [ ]:
movie_25 = pd.read_excel("data/2025년도 영화별 흥행지표_ver4.xlsx") # 25년도 영화별 흥행지표 불러오기

In [ ]:
movie_25.head() #데이터 어떻게 생겼는지 확인인

In [ ]:
movie_24 = pd.read_excel("data/2024년도 영화별 흥행지표.ver6.xlsx")

In [ ]:
movie_24.head(5)

In [ ]:
movie_23 = pd.read_excel("data/2023년도 영화별 흥행지표_ver3.xlsx")

In [ ]:
movie_22 = pd.read_excel("data/2022년도 영화별 흥행지표_ver3.xlsx")

In [ ]:
concat_data = pd.concat([movie_22, movie_23, movie_24, movie_25], axis=0) # 4년도의 영화별 흥행지표 데이터를 합침

In [ ]:
concat_data.head(5)

In [ ]:
concat_data['개봉일'] = pd.to_datetime(concat_data['개봉일'], format='%Y-%m-%d') # 개봉일 컬럼을 datetime형식으로 바꿔줌
# 배우들의 개봉 전달의 점수를 활용하기위해서 datetime형태로 바꿔주었음

In [ ]:
concat_data

In [ ]:
actor22_25 = pd.read_excel("data/2022-2025 영화배우 브랜드 점수.xlsx") # 22년-25년도 영화배우별 브랜드점수

In [ ]:
actor22_25.head(5)

In [ ]:
actor22_25['연도'] = actor22_25['연도'].astype('object') # 연도의 datetype이 float형태이므로 datetime으로 변환하기 위해 object형태로 바꾸고
actor22_25['연도'] = pd.to_datetime(actor22_25['연도'], format = "%Y.%m") # datetime 형태로 바꿔줌

In [ ]:
# 배우들의 전달 점수를 확인하기 위한 함수 (해당 월 1일)
concat_data['brand_period'] = (
    (concat_data['개봉일'] - pd.DateOffset(months=1))
    .dt.to_period('M')
    .dt.to_timestamp()
)

# 배우별 브랜드 점수 조회회
def get_brand_score(row, actor_col):
    act = row[actor_col]
    if pd.isna(act):
        return pd.NA
    sel = actor22_25.loc[
        (actor22_25['이름'] == act) & 
        (actor22_25['연도'] == row['brand_period']), 
        '브랜드평판지수'
    ]
    return sel.iloc[0] if not sel.empty else pd.NA

# 주연1~3에 대해 평판지수 컬럼 추가
for col in ['주연1', '주연2', '주연3','Unnamed: 10','Unnamed: 11','Unnamed: 12','Unnamed: 13','Unnamed: 14','Unnamed: 15','Unnamed: 16']:
    concat_data[f'{col}_브랜드평판'] = concat_data.apply(get_brand_score, actor_col=col, axis=1)

# 보기
cols_to_show = ['영화제목', '개봉일', 'brand_period'] + [f'{col}_브랜드평판' for col in ['주연1','주연2','주연3']]
print(concat_data[cols_to_show].head())


In [ ]:
concat_data.columns

In [ ]:
concat_data.info()

In [ ]:
concat_data = concat_data.fillna(0) # 결측값들을 0으로 채움

cols = ['주연1_브랜드평판', '주연2_브랜드평판', '주연3_브랜드평판',
        'Unnamed: 10_브랜드평판', 'Unnamed: 11_브랜드평판', 'Unnamed: 12_브랜드평판',
        'Unnamed: 13_브랜드평판', 'Unnamed: 14_브랜드평판', 'Unnamed: 15_브랜드평판',
        'Unnamed: 16_브랜드평판']

# 각 행에 대해 0이 아닌 값이 하나라도 있는 경우 필터링 -> 한 행에 다 0인 경우는 제외하기위한 작업
filtered = concat_data[(concat_data[cols] != 0).any(axis=1)]

In [ ]:
filtered

In [ ]:
filtered_score = filtered[cols] # 배우 평판 점수만 보기
filtered_score

In [ ]:
# 상위 3명의 브랜드평판지수
def custom_avg(row):
    # 숫자로 변환하고, 0과 NaN 제외
    values = pd.to_numeric(row, errors='coerce')
    values = values[(values != 0) & (~values.isna())]
    
    count = len(values)
    if count == 0:
        return np.nan
    elif count == 1:
        return values.iloc[0]
    elif count == 2:
        return values.mean()
    else:  # count >= 3
        return values.nlargest(3).mean() # 상위 배우들이 3명이 안될경우를 대비한 코드

# 각 행에 대해 평균 계산
filtered_score['상위3명의 평균'] = filtered_score.apply(custom_avg, axis=1)
filtered_score

In [ ]:
merged_df = pd.concat([filtered, filtered_score], axis=1)


merged_df

In [ ]:
merged_df.columns

In [ ]:
data = merged_df.drop(columns=['Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13',
       'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17',
       'brand_period', '주연1_브랜드평판', '주연2_브랜드평판', '주연3_브랜드평판',
       'Unnamed: 10_브랜드평판', 'Unnamed: 11_브랜드평판', 'Unnamed: 12_브랜드평판',
       'Unnamed: 13_브랜드평판', 'Unnamed: 14_브랜드평판', 'Unnamed: 15_브랜드평판',
       'Unnamed: 16_브랜드평판', '주연1_브랜드평판', '주연2_브랜드평판', '주연3_브랜드평판',
       'Unnamed: 10_브랜드평판', 'Unnamed: 11_브랜드평판', 'Unnamed: 12_브랜드평판',
       'Unnamed: 13_브랜드평판', 'Unnamed: 14_브랜드평판', 'Unnamed: 15_브랜드평판',
       'Unnamed: 16_브랜드평판']) # 필요없는 컬럼들 삭제

In [ ]:
# 상위 3명의 점수의 합
filtered_score['Top3합'] = filtered_score[cols].apply(lambda row: row.nlargest(3).sum(), axis=1)
# 'Top3합' 컬럼을 정수형(int)으로 변환
data['Top3합'] = filtered_score['Top3합'].astype(int) 

In [ ]:
# '상위3명의 평균' 컬럼을 정수형(int)으로 변환
data['상위3명의 평균'] = data['상위3명의 평균'].astype(int)


In [ ]:
data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams['font.family'] = 'Malgun Gothic'  # 한글폰트 설정
plt.rcParams['axes.unicode_minus'] = False     # 마이너스 깨짐 방지


In [ ]:
y = data['관객수']
x = data['상위3명의 평균']

In [ ]:
# 주연배우의 브랜드평판지수 평균과 관객수와의 산점도 그리기
fig, ax = plt.subplots(figsize = (3,3))
ax.scatter(x,y)
ax.set_title('주연배우의 브랜드평판지수 평균과 관객수의 관계')
ax.set_xlabel(('상위3명의평균'))
ax.set_ylabel(('관객수'))


plt.show()

산점도 하단에 점이 몰려 있어 관객수 기준으로 하위(300만 이하), 중간(300만 초과~800만 이하), 상위(800만 초과)를 나누어 살펴봤습니다.

In [ ]:
sort_data = data.sort_values(by=['매출액', '관객수'], ascending=[False, False]) # 매출액과 관객수를 기준으로 내림차순 정렬
sort_data.reset_index(drop=True, inplace=True)

# 하위에 몰려있는 50개의 데이터와 중위 6개, 상위 4개 데이터 분리 (선형관계 발견x)
def classify(row):
    if row['관객수'] <= 3000000:  # 100,000 ~ 3,000,000 사이
        return '하위'
    elif row['관객수'] <= 8000000:  # 4,000,000 ~ 8,000,000 사이
        return '중간'
    else:  # 10,000,000 이상
        return '상위'

sort_data['분위'] = sort_data.apply(classify, axis=1)

# 분위별로 새로운 데이터프레임 만들기
df_lower = sort_data[sort_data['분위'] == '하위']
df_middle = sort_data[sort_data['분위'] == '중간']
df_upper = sort_data[sort_data['분위'] == '상위']


# 백분위수 (상위20%에서 어느정도 선형관계 발견)
top_20 = sort_data.iloc[:12].copy()       # 상위 20%
next_30 = sort_data.iloc[12:30].copy()    # 그 다음 30%
bottom_50 = sort_data.iloc[30:].copy()    # 하위 50%

# 산점도 출력
y = bottom_50['관객수']
x = bottom_50['Top3합']

fig, ax = plt.subplots(figsize = (5,5))
ax.scatter(x,y)
ax.set_title('lower 영화')
ax.set_xlabel(('Top3합'))
ax.set_ylabel(('관객수'))


plt.show()

산점도 파악결과 눈으로는 상관관계를 파악하기 어렵다고 판단되어, 상관계수와 p밸류값을 확인 해보았음

In [ ]:
from scipy.stats import pearsonr

correlation, p_value = pearsonr(data['상위3명의 평균'], data['관객수'])

In [ ]:
correlation

In [ ]:
p_value

피어슨 상관계수는 약 0.4481, p값은 약 0.0003306입니다. 분석 표본에서 양의 선형 관계가 관측됐습니다.

사실 영화 흥행요소가 매우 많기때문에 이런 결과가 나왔다고 생각했고, 그래서 다른 요인들을 시각화 해보기로함

In [ ]:
import statsmodels.api as sm
import matplotlib.pyplot as plt

# 회귀분석


X = data['상위3명의 평균']
y = data['관객수']
X_const = sm.add_constant(X)

# 회귀 분석 모델 생성 및 학습
model = sm.OLS(y, X_const).fit()

# 회귀선 예측값 생성
y_pred = model.predict(X_const)

plt.figure(figsize=(10, 6))
plt.scatter(X, y, color='skyblue', label='실제 관객수')
plt.plot(X, y_pred, color='red', label='회귀 추세선')

# 각 점에 영화제목 표시
for i, row in data.iterrows():
    plt.text(row['상위3명의 평균'], row['관객수'], row['영화제목'], fontsize=8, ha='left', va='bottom')

plt.xlabel('상위3명의 평균 브랜드 평판지수')
plt.ylabel('관객수')
plt.title('브랜드 평판 점수와 관객수와의 관계')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# 6. 회귀 결과 요약 출력
print(model.summary())


결정계수는 0.201, 수정 결정계수는 0.187입니다. 회귀 F 검정의 p값은 약 0.000331입니다.

In [ ]:
data['매출액'].mean() #매출액 평균 보기

In [ ]:
data['매출액'].var() #매출액 분산 보기

In [ ]:
data['매출액'].quantile([0.25,0.5,0.75]) # 매출액의 상위 25%, 하위 25%보기 (백분위수)

In [ ]:
sb.boxplot(y='매출액',data=data[['매출액']]) #박스플롯
plt.show()

매출액 분포는 높은 값 방향으로 치우쳐 있으며, 일부 상위 매출작은 상자그림의 위쪽 수염을 벗어납니다.

In [ ]:
data.plot(kind='scatter',x='매출액',y='관객수') # 매출액과 관객수의 산점도

매출액과 관객수는 뚜렷한 양의 관계가 있는 것으로 파악됨

In [ ]:
xx = data['스크린수']
yy = data['관객수']
# 스크린수와 관객수의 관계 산점도 그리기
fig, ax = plt.subplots(figsize = (3,3))
ax.scatter(xx,yy)
ax.set_title('스크린수와 관객수의 관계')
ax.set_xlabel(('스크린수'))
ax.set_ylabel(('관객수'))


스크린수와 관객수와의 관계는 어느정도 선형관계가 있는 것으로 보임